Edited from: https://www.kaggle.com/code/pavansanagapati/ensemble-learning-techniques-tutorial

# Introduction <a id="1"></a> <br>
    
Suppose you wanted to purchase a car.Now by just visiting the first car company and based on the dealer's advise  will we straight away make a buy on a car? Answer is defenitely a big NO right?
    
![](https://thumbs.dreamstime.com/b/car-sale-4167169.jpg)
    
So what we do is first decide whether which car to buy ,whether it is a new or used car ,type of car,model and year of manufacture, look for list of dealers ,look for discounts/offers ,customer reviews,opinion from friends and family, performance ,fuel efficiency and obvious any car buyer will for the best price range etc.
    
![](https://encrypted-tbn0.gstatic.com/images?q=tbn%3AANd9GcSC0-mqf3xqr3MESGW-mGwaWQkkBjwJGbNFsQ&usqp=CAU)
    
 In short, you wouldn’t directly reach a conclusion, but will instead make a decision considering all the above mentioned factors before we decide on the best choice.
    
**Ensemble models** in machine learning operate on a similar idea.
![](https://i.pinimg.com/474x/d7/c7/9b/d7c79b0c7abc5a34e17710fe596f6834.jpg)    
Ensemble Learning helps improve machine learning results by combining several models to improve predictive performance compared to a single model.
![](https://i.imgur.com/L2Jaqm8.png)
    
# Build An Ensemble <a id="2"></a> <br>

## Max Voting / Voting Classifier <a id="2.1"></a> <br>

This notebook introduces a simple but powerful ensemble technique: voting. For those curious about more advanced methods, check out resources:https://www.kaggle.com/code/pavansanagapati/ensemble-learning-techniques-tutorial or https://www.analyticsvidhya.com/blog/2018/06/comprehensive-guide-for-ensemble-models/

Voting is widely used for classification challenges, where several models contribute to the final decision, each casting a "vote" for their prediction. The majority vote decides the final output.

The concept is brought to life with a Voting Classifier, which combines the strengths of multiple models, selecting the class with the highest vote as the prediction. It's an effective way to improve accuracy by leveraging the diversity of the ensemble.

We'll demonstrate this with the Wine dataset, showcasing how a Voting Classifier can enhance prediction performance.



In [1]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from collections import Counter
import numpy as np

seed = 42
np.random.seed(seed)

In [2]:
# we only use first four features as X to demonstrate the difference between models.
wine = load_wine()
X = wine.data[:, :4]
Y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X,Y,test_size = 0.20,random_state = seed)

Now let's create three different models for voting, one LogisticRegression, one SVC and one DecisionTreeClassifier and train with training set.


In [4]:
lr = LogisticRegression(random_state = seed)
lr.fit(X_train, y_train)
svc = SVC(probability=True, random_state = seed)
svc.fit(X_train, y_train)
dt = DecisionTreeClassifier(random_state = seed)
dt.fit(X_train, y_train)
model_pool = [lr,svc,dt]

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Let's see how well each model does by checking their accuracy scores:

In [5]:
y_pred_lr = lr.predict(X_test)
accuracy_score(y_test, y_pred_lr)

0.8333333333333334

In [6]:
y_pred_svc = svc.predict(X_test)
accuracy_score(y_test, y_pred_svc)

0.6111111111111112

In [7]:
y_pred_dt = dt.predict(X_test)
accuracy_score(y_test, y_pred_dt)

0.8055555555555556

## Hard Voting / Soft Voting <a id="2.2"></a> <br>
Hard Voting: This method goes with the majority vote. For example, if three models predict A, A, and B for a class, A wins because it's the majority choice.

Soft Voting: This method averages the probability predictions for each class from all models. If class A gets probabilities of 0.30, 0.47, 0.53 and class B gets 0.20, 0.32, 0.40, class A wins with the higher average probability of 0.4333 compared to B's 0.3067.
![](https://image.slidesharecdn.com/7-180514114334/95/ensemble-learning-and-random-forests-12-638.jpg?cb=1527755412)

In [8]:
def hard_voting(model_pool,X):
    # Collect predictions from all models
    all_preds = np.array([model.predict(X) for model in model_pool]).T

    # Majority vote for each sample
    return np.array([Counter(row).most_common(1)[0][0] for row in all_preds])


accuracy_score(y_test,hard_voting(model_pool,X_test))

0.8333333333333334

In [9]:
def soft_voting(model_pool,X):
    # Collect predicted probabilities from all models
    all_probs = np.array([model.predict_proba(X) for model in model_pool])

    # Average probabilities across models
    avg_probs = np.mean(all_probs, axis=0)

    # Take the class with the highest average probability
    return np.argmax(avg_probs, axis=1)
accuracy_score(y_test,soft_voting(model_pool,X_test))

0.9166666666666666

# Summary

In ensemble learning, **hard voting** combines model predictions by selecting the class with the majority of votes, considering only the **predicted labels** and ignoring model confidence.

In contrast, **soft voting** averages the **predicted probabilities** from all models and chooses the class with the highest mean probability, effectively leveraging each model’s confidence.

In this experiment with Logistic Regression, SVM, and Decision Tree models, soft voting outperformed hard voting because it incorporated the varying confidence levels of the individual models, allowing more certain predictions to influence the final decision even when in the minority.

This is often the case in practice when models have different strengths or produce well-calibrated probabilities.

It is interesting to note that methods like XGBoost implement a similar principle implicitly, as each tree’s contribution is weighted during sequential learning, effectively performing a confidence-informed aggregation of predictions.